# British Airways: Predicting Customer Buying Behaviour

Reproducible analysis for the Forage Data Science Job Simulation.

**Input:** `customer_booking.csv` (place the CSV in the same folder as this notebook).  
**Target:** `booking_complete` (1 = booking completed, 0 = not completed).

This notebook explores the dataset, prepares features, runs 5-fold stratified cross-validation on the training split, evaluates a class-weighted Random Forest on a held-out test split, and visualizes permutation feature importance. The model is a learning exercise, not a production-ready booking decision system.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

In [ ]:
# Load the CSV. Keep customer_booking.csv beside this notebook.
csv_path = Path("customer_booking.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        "Place the Forage Customer Booking.csv dataset in this project folder as customer_booking.csv. "
        "The source dataset is intentionally not included in this repository."
    )

df = pd.read_csv(csv_path, encoding="ISO-8859-1")
print("Shape:", df.shape)
print("\nColumns and data types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values:", int(df.isna().sum().sum()))
print("\nTarget counts:")
display(df["booking_complete"].value_counts().rename(index={0:"No completed booking", 1:"Completed booking"}))
print("\nTarget rate:", round(df["booking_complete"].mean() * 100, 2), "%")
display(df.head())

## 1. Explore and prepare the data

The supplied starter notebook notes that there are no null values. We verify that in this file. The booking target is imbalanced, so accuracy alone is not a sufficient metric.

Feature engineering used here:
- Convert `flight_day` to a consistent weekday number, following the starter notebook.
- Create `purchase_lead_band`, `length_of_stay_band`, and `departure_period` features to test broad, interpretable groups.
- One-hot encode categorical variables inside a pipeline so validation folds do not learn encoding from held-out data.

In [ ]:
print(df.describe(include="all").transpose().to_string())

day_map = {"Mon": 1, "Tue": 2, "Wed": 3, "Thu": 4, "Fri": 5, "Sat": 6, "Sun": 7}
df["flight_day_num"] = df["flight_day"].map(day_map)

df["purchase_lead_band"] = pd.cut(
    df["purchase_lead"], bins=[-1, 7, 30, 90, np.inf],
    labels=["0-7d", "8-30d", "31-90d", ">90d"]
)
df["length_of_stay_band"] = pd.cut(
    df["length_of_stay"], bins=[-1, 3, 7, 14, np.inf],
    labels=["0-3d", "4-7d", "8-14d", ">14d"]
)
df["departure_period"] = pd.cut(
    df["flight_hour"], bins=[-1, 5, 11, 17, 23],
    labels=["Overnight/early", "Morning", "Afternoon", "Evening"],
    include_lowest=True
)

target = "booking_complete"
X = df.drop(columns=[target, "flight_day"])
y = df[target].astype(int)

cat_cols = [
    "sales_channel", "trip_type", "route", "booking_origin",
    "purchase_lead_band", "length_of_stay_band", "departure_period"
]
num_cols = [c for c in X.columns if c not in cat_cols]

print("Features:", len(X.columns))
print("Categorical features:", cat_cols)
print("Numerical features:", num_cols)

In [ ]:
# Stratified holdout split preserves the target-class ratio in train and test data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

model = RandomForestClassifier(
    n_estimators=180,
    max_depth=22,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

pipe = Pipeline([("preprocess", preprocess), ("model", model)])
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

## 2. Cross-validation

Five-fold stratified cross-validation is run on the training split. Average precision is included because only about 15% of rows are positive (completed bookings); this makes ranking performance on the positive class important.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
}

cv_results = cross_validate(
    pipe, X_train, y_train, cv=cv, scoring=scoring,
    n_jobs=1, return_train_score=False
)

cv_summary = pd.DataFrame({
    "Metric": [key.replace("test_", "").replace("_", " ").title()
               for key in cv_results if key.startswith("test_")],
    "Mean": [cv_results[key].mean() for key in cv_results if key.startswith("test_")],
    "Std. dev. across folds": [cv_results[key].std() for key in cv_results if key.startswith("test_")],
})
print(cv_summary.round(3).to_string(index=False))

## 3. Holdout test evaluation

The class-weighted model favors finding more likely bookers, so recall is prioritized over raw accuracy. This comes at a precision trade-off: many customers flagged by the model may not complete a booking. The decision threshold should ultimately reflect campaign capacity and the relative costs of missed bookings versus unnecessary outreach.

In [ ]:
pipe.fit(X_train, y_train)
y_prob = pipe.predict_proba(X_test)[:, 1]
threshold = 0.50
y_pred = (y_prob >= threshold).astype(int)

holdout_metrics = pd.Series({
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "Average precision": average_precision_score(y_test, y_prob),
    "Actual booking rate": y_test.mean(),
    "Predicted positive rate": y_pred.mean(),
})
print(holdout_metrics.to_frame("Value").round(3).to_string())
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=3, zero_division=0))

## 4. Feature importance

Permutation importance measures how much the held-out average precision decreases when one input feature is shuffled. This works at the original feature level (rather than separately reporting each one-hot encoded category). It identifies predictive associations, not causal effects. Importance can also vary by sample and model choice.

In [ ]:
# Use a fixed random sample of the holdout data to make feature-importance estimation repeatable.
n_perm = min(3000, len(X_test))
X_perm = X_test.sample(n=n_perm, random_state=RANDOM_STATE)
y_perm = y_test.loc[X_perm.index]

perm = permutation_importance(
    pipe, X_perm, y_perm, n_repeats=3, random_state=RANDOM_STATE,
    scoring="average_precision", n_jobs=1
)
importance = pd.DataFrame({
    "Feature": X_perm.columns,
    "Mean drop in average precision": perm.importances_mean,
    "Std. dev.": perm.importances_std,
}).sort_values("Mean drop in average precision", ascending=False)
print(importance.head(10).round(4).to_string(index=False))

top = importance.head(8).sort_values("Mean drop in average precision")
plt.figure(figsize=(8, 5))
plt.barh(top["Feature"], top["Mean drop in average precision"], color="#C8102E")
plt.xlabel("Decrease in average precision when feature is shuffled")
plt.title("Feature permutation importance", loc="left", weight="bold")
plt.grid(axis="x", alpha=0.2)
plt.gca().set_axisbelow(True)
plt.tight_layout()
plt.show()

## 5. Findings and business interpretation

Results from the recorded local run (small differences can occur across software versions and reruns):

- Approximately **14.96%** of records have `booking_complete = 1`, so the target is imbalanced.
- Five-fold training cross-validation achieved mean **ROC-AUC = 0.776** and **average precision = 0.359**.
- On the held-out test set at a 0.50 threshold, the model achieved **69.4% accuracy**, **72.1% balanced accuracy**, **29.6% precision**, **75.9% recall**, **42.6% F1-score**, and **0.784 ROC-AUC**.
- Permutation importance ranked `booking_origin` first, followed by `route`, `length_of_stay_band`, and `wants_extra_baggage` in this model configuration.

**Recommendation:** Use model scores to prioritize a pilot campaign, tune the cutoff against campaign capacity and the cost of false positives, and validate on newer booking cohorts. Feature importance reflects predictive association, not causation.
